In [7]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

In [8]:
ITEM_MASTER_PATH = Path("../data/item_master.csv")
PROMO_CALENDAR_PATH = Path("../data/promo_calendar.csv")
SALES_HISTORY_PATH = Path("../data/sales_history.csv")
STOCK_HISTORY_PATH = Path("../data/stock_history.csv")

In [9]:
item_master_df = pd.read_csv(ITEM_MASTER_PATH)
promo_calendar_df = pd.read_csv(PROMO_CALENDAR_PATH)
sales_history_df = pd.read_csv(SALES_HISTORY_PATH)
stock_history_df = pd.read_csv(STOCK_HISTORY_PATH)

In [10]:
sales_history_df["sku"] = sales_history_df["sku"].astype(str)
stock_history_df["sku"] = stock_history_df["sku"].astype(str)
item_master_df["sku"] = item_master_df["sku"].astype(str)
promo_calendar_df["sku"] = promo_calendar_df["sku"].astype(str)

In [11]:
full_df = (
    sales_history_df.merge(stock_history_df, on=["sku", "period", "location"], how="outer")
    .merge(promo_calendar_df, on=["sku", "period"], how="left")
    .merge(item_master_df, on="sku", how="left")
)
full_df

,sku,location,period,qty,stock_end_qty,days_out_of_stock,promo_type,discount_pct,category,abc_class,launch_period,predecessor_sku,uom
0,SKU_0001,MSK,2023-01,243,174,0,NaN,NaN,SNACK,B,2023-01,NaN,PCS
1,SKU_0001,MSK,2023-02,179,150,0,NaN,NaN,SNACK,B,2023-01,NaN,PCS
2,SKU_0001,MSK,2023-03,168,193,0,NaN,NaN,SNACK,B,2023-01,NaN,PCS
3,SKU_0001,MSK,2023-04,204,232,0,NaN,NaN,SNACK,B,2023-01,NaN,PCS
4,SKU_0001,MSK,2023-05,153,199,0,NaN,NaN,SNACK,B,2023-01,NaN,PCS
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1521,SKU_0040,MSK,2025-08,0,1,0,NaN,NaN,BEVERAGE,B,2023-01,NaN,PCS
1522,SKU_0040,MSK,2025-09,27,41,0,NaN,NaN,BEVERAGE,B,2023-01,NaN,PCS
1523,SKU_0040,MSK,2025-10,0,0,0,NaN,NaN,BEVERAGE,B,2023-01,NaN,PCS
1524,SKU_0040,MSK,2025-11,21,12,0,NaN,NaN,BEVERAGE,B,2023-01,NaN,PCS


In [12]:
full_df.to_csv("../data/full_df.csv", index=False)

In [7]:
def clean_full_duplicates(df: pd.DataFrame) -> pd.DataFrame:
    df_copy = df.copy()
    df_copy = df_copy.drop_duplicates(subset=["sku", "period", "location"])
    return df_copy
df_no_duplicates = clean_full_duplicates(full_df)

In [8]:
def fix_null_sells(df: pd.DataFrame) -> pd.DataFrame:
    df_copy = df.copy()
    df_copy["qty"] = np.where(df_copy["qty"] > 0, df_copy["qty"], 0)
    return df_copy
df_no_null_sells = fix_null_sells(df_no_duplicates)

In [9]:
def detect_uom_shift(df: pd.DataFrame, window_size:int=3, threshold:int=3) -> pd.DataFrame:
    df_copy = df.copy()
    df_copy = df_copy.sort_values(["sku", "period"])
    shifts = []
    for sku, group in df.groupby('sku'):
        ts = group.set_index('period')['qty']
        
        med_before = ts.rolling(window=window_size, min_periods=1).median()
        
        med_after = ts.iloc[::-1].rolling(window=window_size, min_periods=1).median().iloc[::-1]
        
        med_after_shifted = med_after.shift(-1)
        
        ratio = med_after_shifted / (med_before + 1)
        
        anomalies = ratio[ratio > threshold]
        
        if not anomalies.empty:
            shift_date = anomalies.index[0] + pd.DateOffset(months=1) 
            
            shifts.append({
                'sku': sku,
                'shift_start_period': shift_date.strftime('%Y-%m') if isinstance(shift_date, pd.Timestamp) else shift_date,
                'jump_ratio': round(anomalies.iloc[0], 1)
            })
            
    return pd.DataFrame(shifts)

In [10]:
df_no_null_sells["period"] = pd.to_datetime(df_no_null_sells["period"])
deteced_shifts = detect_uom_shift(df_no_null_sells)
deteced_shifts

,sku,shift_start_period,jump_ratio
0,SKU_0002,2023-10,26.0
1,SKU_0007,2023-11,1550.0
2,SKU_0009,2024-06,6.0
3,SKU_0012,2024-11,631.0
4,SKU_0015,2024-03,8.0
5,SKU_0017,2023-02,3.7
6,SKU_0018,2024-06,1309.9
7,SKU_0024,2024-06,10.0
8,SKU_0031,2023-08,96.0
9,SKU_0033,2024-05,2758.0


In [ ]:
def merge_predecessors(df: pd.DataFrame) -> pd.DataFrame:
    """
    Приклеивает историю продаж предшественника (old_sku) к новому товару (new_sku).
    Старый товар после этого удаляется из датасета.
    """
    df_merged = df.copy()
    
    # 1. Находим все связки: новый товар -> старый товар
    # (Берем только строки, где есть predecessor_sku)
    mapping = df_merged[['sku', 'predecessor_sku']].dropna().drop_duplicates()
    
    skus_to_drop = []
    
    for _, row in mapping.iterrows():
        new_sku = row['sku']
        old_sku = row['predecessor_sku']
        
        # Выделяем историю старого товара
        old_history = df_merged[df_merged['sku'] == old_sku].copy()
        
        if not old_history.empty:
            # Находим первый месяц продаж НОВОГО товара
            new_sku_first_period = df_merged[df_merged['sku'] == new_sku]['period'].min()
            
            # Отрезаем историю старого товара, оставляя только месяцы до запуска нового
            # (защита от задвоения продаж в переходный период)
            old_history = old_history[old_history['period'] < new_sku_first_period]
            
            # Получаем атрибуты нового товара (категория, abc_class и т.д.)
            new_sku_metadata = df_merged[df_merged['sku'] == new_sku].iloc[0]
            
            # Подменяем SKU и выравниваем справочные признаки
            old_history['sku'] = new_sku
            static_cols = ['category', 'abc_class', 'launch_period', 'predecessor_sku', 'uom']
            for col in static_cols:
                if col in old_history.columns:
                    old_history[col] = new_sku_metadata[col]
            
            # Добавляем старый SKU в список на удаление
            skus_to_drop.append(old_sku)
            
            # Приклеиваем переименованную историю
            df_merged = pd.concat([df_merged, old_history], ignore_index=True)
            
    # 2. Очищаем датасет от снятых с производства старых SKU
    if skus_to_drop:
        df_merged = df_merged[~df_merged['sku'].isin(skus_to_drop)]
        
    return df_merged.sort_values(['sku', 'location', 'period']).reset_index(drop=True)

In [2]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder()

ModuleNotFoundError: No module named 'src'